# Laboratório 10 — O Pipeline Definitivo

**RAG + QLoRA + FlashAttention-2 + KV Cache na GPU**

Disciplina: Tópicos em IA — Instituto de Ensino Superior iCEV

Aluno: Vinicius Henrique Albino Andrade

---

## Contexto
A HealthTech (Lab 09) colocou em produção um RAG médico que recupera ~5 capítulos inteiros de manuais (≈15k tokens) e injeta num LLM fine-tunado. O servidor estourou a VRAM por causa do custo O(n²) do Self-Attention.

Missão: orquestrar **QLoRA 4-bit** (Unidade II) + **KV Cache** + **FlashAttention-2** (Unidade I) para evitar OOM e ainda reduzir latência.

## Requisito de runtime
- **GPU NVIDIA com CUDA** (obrigatório por causa do `bitsandbytes`).
- FlashAttention-2 oficialmente exige **Ampere (sm_80) ou superior** (L4, A100, A10, H100). Se cair em T4 (Turing sm_75), o notebook faz **fallback automático para `sdpa`** — PyTorch Scaled Dot Product Attention — que também usa kernels memory-efficient e preserva o ganho de memória da fase de prompting, embora não seja exatamente o FA-2.

## Setup — instalação de dependências

Execute esta célula uma vez por sessão de Colab.

In [ ]:
!pip install -q -U "transformers>=4.43" "accelerate>=0.31" "bitsandbytes>=0.43" "sentencepiece" "matplotlib"
# FlashAttention-2 (pode falhar em Turing/T4 — tratamos via fallback adiante)
!pip install -q "flash-attn>=2.5.0" --no-build-isolation || echo "flash-attn não instalou — usaremos SDPA como fallback documentado."

In [ ]:
import os, gc, time, json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

assert torch.cuda.is_available(), "GPU CUDA é obrigatória para este laboratório."

GPU_NAME = torch.cuda.get_device_name(0)
CAPABILITY = torch.cuda.get_device_capability(0)
SUPPORTS_FA2 = CAPABILITY[0] >= 8  # Ampere ou superior

print(f"GPU detectada : {GPU_NAME}")
print(f"Compute capab.: {CAPABILITY}  (sm_{CAPABILITY[0]}{CAPABILITY[1]})")
print(f"Suporta FA-2  : {SUPPORTS_FA2}")
print(f"VRAM total    : {torch.cuda.get_device_properties(0).total_memory / 1024**2:.0f} MB")

---

## Passo 1 — Ingestão Eficiente (QLoRA 4-bit via bitsandbytes)

Carregar o LLM base em FP16 ocuparia ~2.2 GB só para o TinyLlama-1.1B. Em modelos maiores isso já estoura VRAM. **QLoRA quantiza os pesos do modelo congelado para 4 bits (NF4)** e mantém o cálculo em FP16, reduzindo a pegada de memória em ~4× sem perda significativa de qualidade.

Configuração usada:
- `load_in_4bit=True`
- `bnb_4bit_quant_type="nf4"` (4-bit NormalFloat — distribuição otimizada para pesos)
- `bnb_4bit_compute_dtype=torch.float16`
- `bnb_4bit_use_double_quant=True` (quantiza também as constantes de quantização → economia extra)

In [ ]:
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
vram_before = torch.cuda.memory_allocated() / 1024**2

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()

vram_after = torch.cuda.memory_allocated() / 1024**2
vram_model_only = vram_after - vram_before

print(f"VRAM antes do load        : {vram_before:.2f} MB")
print(f"VRAM após carregar modelo : {vram_after:.2f} MB")
print(f"Pegada do modelo quantizado: {vram_model_only:.2f} MB")
print(f"\n[MÉTRICA — PASSO 1] Modelo {MODEL_ID} em 4-bit NF4 ocupa ~{vram_model_only:.0f} MB de VRAM.")